# Reverse-engineering the scores

Flip the model: predict `EXT_SOURCE` from the internal features. High reconstruction R² means the purchased score is largely rebuildable from data the lender already owns; low R² means it carries information from outside the lender's walls.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score
from src.data import load_selected, RAW

X, y = load_selected()
ext = [c for c in X.columns if "ext_source" in c.lower() or c.lower().startswith("ext_calc_")]
internal = [c for c in X.columns if c not in ext]
raw_ext = pd.read_csv(RAW / "application_train.csv",
                      usecols=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]).set_index("SK_ID_CURR").reindex(X.index)

def reg():
    return lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8,
                             colsample_bytree=0.7, subsample_freq=1, n_jobs=-1, verbose=-1)
raw_ext.notna().mean().round(3)

EXT_SOURCE_1    0.436
EXT_SOURCE_2    0.998
EXT_SOURCE_3    0.802
dtype: float64

## How reconstructable is each score?

In [2]:
kf = KFold(5, shuffle=True, random_state=42)
for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    m = raw_ext[c].notna().to_numpy()
    pred = cross_val_predict(reg(), X.loc[m, internal], raw_ext[c][m], cv=kf)
    print(f"{c}:  R2 {r2_score(raw_ext[c][m], pred):+.3f}   (n={int(m.sum())}, coverage {m.mean():.2f})")

EXT_SOURCE_1:  R2 +0.493   (n=134133, coverage 0.44)
EXT_SOURCE_2:  R2 +0.170   (n=306851, coverage 1.00)
EXT_SOURCE_3:  R2 +0.516   (n=246546, coverage 0.80)


## What is the score made of?

In [3]:
c = "EXT_SOURCE_2"
m = raw_ext[c].notna()
model = reg().fit(X.loc[m, internal], raw_ext[c][m])
samp = X.loc[m, internal].sample(min(5000, int(m.sum())), random_state=0)
sv = model.booster_.predict(samp, pred_contrib=True)[:, :-1]
pd.Series(np.abs(sv).mean(0), index=internal).sort_values(ascending=False).head(15)

REGION_RATING_CLIENT                    0.017114
DAYS_BIRTH                              0.016768
prev_HOUR_APPR_PROCESS_START_mean       0.015056
NAME_EDUCATION_TYPE_Higher_education    0.008677
REGION_RATING_CLIENT_W_CITY             0.007029
DAYS_EMPLOYED                           0.006480
bureau_DAYS_CREDIT_mean                 0.004629
x_ae_13                                 0.004093
AMT_GOODS_PRICE                         0.004062
income_to_age                           0.003809
prev_HOUR_APPR_PROCESS_START_max        0.003246
credit_to_goods                         0.003139
YEARS_BEGINEXPLUATATION_MEDI            0.002914
ELEVATORS_AVG                           0.002895
OWN_CAR_AGE                             0.002709
dtype: float64

## What each score is made of

Reconstruct all three scores and read the top internal drivers of each. The two rebuildable scores (1 and 3) should lean on obvious internal proxies; the least rebuildable (2) should look thinner, since more of it lives outside the lender's data.

In [4]:
for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    mk = raw_ext[c].notna()
    mdl = reg().fit(X.loc[mk, internal], raw_ext[c][mk])
    s = X.loc[mk, internal].sample(min(4000, int(mk.sum())), random_state=0)
    contrib = mdl.booster_.predict(s, pred_contrib=True)[:, :-1]
    top = pd.Series(np.abs(contrib).mean(0), index=internal).sort_values(ascending=False).head(6)
    print(f"\n{c}   top internal drivers")
    print(top.round(3).to_string())


EXT_SOURCE_1   top internal drivers
DAYS_BIRTH                                         0.095
CODE_GENDER_M                                      0.026
CODE_GENDER_F                                      0.022
NAME_EDUCATION_TYPE_Higher_education               0.014
NAME_EDUCATION_TYPE_Secondary_secondary_special    0.009
NAME_FAMILY_STATUS_Married                         0.008

EXT_SOURCE_2   top internal drivers
REGION_RATING_CLIENT                    0.017
DAYS_BIRTH                              0.017
prev_HOUR_APPR_PROCESS_START_mean       0.015
NAME_EDUCATION_TYPE_Higher_education    0.009
REGION_RATING_CLIENT_W_CITY             0.007
DAYS_EMPLOYED                           0.006

EXT_SOURCE_3   top internal drivers
bureau_DAYS_CREDIT_mean                   0.022
bureau_DAYS_CREDIT_max                    0.021
bureau_CREDIT_TYPE_Credit_card_sum        0.017
bureau_active_DAYS_CREDIT_max             0.016
x_total_overdue_to_income                 0.016
bureau_active_AMT_CREDIT_SUM_DE